# Factoring Polynomials — Greatest Common Factor (GCF)

Notebook 18 covered multiplying and dividing polynomials — including multiplying a monomial by a polynomial by distributing it over every term. Factoring is that operation run in reverse: given an already-expanded polynomial, find the largest monomial that was distributed to produce it, and pull it back out.

This notebook covers the first and most fundamental factoring technique: factoring out the **greatest common factor (GCF)** shared by every term. It is the technique you check first, before any other factoring method — every polynomial should be checked for a common factor before anything else is attempted.

**Prerequisites:** Polynomial Operations — Adding and Subtracting (15), Multiplying and Dividing Monomials (17), Multiplying and Dividing Polynomials (18) — particularly section 1 (monomial × polynomial) and section 5 (÷ monomial), which this notebook runs in reverse.

In [ ]:
import sys
sys.path.insert(0, '../..')  # repo root, so `shared` is importable

# poly_str/poly_eval/poly_clean (15), poly_mul_mono/poly_div_mono (18),
# and mono_str (17) were taught in earlier notebooks -- imported here, not re-taught
from shared.polynomials import (
    poly_str, poly_eval, poly_clean,
    poly_mul_mono, poly_div_mono,
    mono_str,
)
from math import gcd
from functools import reduce

# --- New for this notebook ---
def coeff_gcf(coeffs):
    """Greatest common factor of a list of integer coefficients (sign-agnostic)."""
    return reduce(gcd, (abs(c) for c in coeffs))

def poly_gcf(poly):
    """Return (gcf_coeff, gcf_exp) -- the GCF of a polynomial's terms, as a monomial."""
    coeffs = list(poly.values())
    exps = list(poly.keys())
    g_coeff = coeff_gcf(coeffs)
    g_exp = min(exps)          # the variable part every term can spare -- the SMALLEST exponent present
    leading_exp = max(exps)
    if poly[leading_exp] < 0:  # convention: factor out a negative so the remaining leading term is positive
        g_coeff = -g_coeff
    return g_coeff, g_exp

def factor_gcf(poly):
    """Factor a polynomial as gcf * remaining. Returns (gcf_coeff, gcf_exp, remaining_poly)."""
    g_coeff, g_exp = poly_gcf(poly)
    remaining = poly_div_mono(poly, g_coeff, g_exp)
    return g_coeff, g_exp, remaining

# Quick smoke test
p = {3: 12, 2: 18, 1: 24}
gc, ge, rem = factor_gcf(p)
print(f'Example: {poly_str(p)}  =  {mono_str(gc, ge)}({poly_str(rem)})')


---

## 1. Try It First — The Biggest Tile That Fits

Before any rule: picture the coefficients of $12x^3 + 18x^2 + 24x$ as three bars, 12, 18, and 24 units long. Question — what is the biggest ruler you could lay down that measures *all three bars exactly*, with no leftover bit at the end of any of them?

Try a few ruler lengths and see which ones work.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

coeffs = [12, 18, 24]
labels = ['12x^3', '18x^2', '24x']
candidates = [2, 3, 4, 6, 8, 12]   # ruler lengths to try, small to large

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
fig.suptitle('Sliding a candidate ruler length across the three bars', fontsize=13, fontweight='bold')

bar_color = '#89b4fa'
fail_color = '#f38ba8'
pass_color = '#a6e3a1'

for ax, cand in zip(axes.flat, candidates):
    divides_all = all(n % cand == 0 for n in coeffs)
    color = pass_color if divides_all else fail_color
    for i, n in enumerate(coeffs):
        ax.barh(i, n, color=bar_color, edgecolor='white', height=0.6)
        # draw gridlines at every multiple of the candidate ruler length
        for tick in range(0, n + 1, cand):
            ax.axvline(tick, ymin=(i)/3 + 0.05, ymax=(i+1)/3 - 0.05, color=color, linewidth=2)
    ax.set_yticks(range(3))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlim(0, 26)
    status = 'fits evenly' if divides_all else 'leftover -- does not fit'
    ax.set_title(f'ruler = {cand}   ({status})', fontsize=10,
                 color='#40a02b' if divides_all else '#d20f39')
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print('Candidate ruler lengths and whether each divides all three coefficients evenly:')
best = 1
for cand in range(1, 13):
    ok = all(n % cand == 0 for n in coeffs)
    if ok:
        best = cand
    print(f'  ruler = {cand:>2}   divides 12, 18, and 24 evenly?  {ok}')
print(f'\nLargest ruler that fits all three exactly: {best}  <-- this is the GCF of 12, 18, 24')


---

## 2. What Just Happened

The biggest ruler that measured all three bars exactly — 6 — is the **greatest common factor (GCF)** of 12, 18, and 24. Every one of the three coefficients is a whole-number multiple of 6, and no bigger number has that property.

For a polynomial, the GCF is not just a number — it is a **monomial**: a numeric part (the GCF of the coefficients) and a variable part (the largest power of $x$ every term can spare). Finding it and pulling it out is exactly the reverse of the "monomial × polynomial" distribution from notebook 18:

$$12x^3 + 18x^2 + 24x \;=\; 6x \cdot (2x^2 + 3x + 4)$$

Multiply the right side back out and you get the left side — factoring and multiplying undo each other, the same way division and multiplication do.

The two parts of the GCF — the number, and the power of $x$ — are worth understanding one at a time before combining them.

---

## 3. Isolating the Numeric Part

Hold the variable structure fixed (three terms, degrees 3, 2, 1 every time) and vary only the coefficients. The numeric part of the GCF is just `coeff_gcf` from the helpers above — the same idea as reducing a fraction to lowest terms (notebook 04).

In [ ]:
def show_factor(poly):
    """Print the factoring process step by step."""
    coeffs = list(poly.values())
    exps = list(poly.keys())
    g_coeff, g_exp = poly_gcf(poly)
    print(f'  {poly_str(poly)}')
    print(f'    coefficients: {coeffs}  ->  numeric GCF = {coeff_gcf([abs(c) for c in coeffs])}')
    print(f'    exponents:    {sorted(exps)}  ->  variable GCF = {mono_str(1, min(exps))}')
    remaining = poly_div_mono(poly, g_coeff, g_exp)
    reconstructed = poly_mul_mono(remaining, g_coeff, g_exp)
    x = 5
    orig, check = poly_eval(poly, x), poly_eval(reconstructed, x)
    print(f'    GCF = {mono_str(g_coeff, g_exp)}   ->   {mono_str(g_coeff, g_exp)}({poly_str(remaining)})')
    print(f'    Verify at x={x}: {orig} = {check}  {"OK" if orig == check else "MISMATCH"}')
    print()

show_factor({3: 12, 2: 18, 1: 24})   # numeric GCF = 6
show_factor({3: 12, 2: 16, 1:  8})   # same shape, numeric GCF = 4


---

## 4. Isolating the Variable Part

Now hold the coefficients fixed at 1 (so the numeric GCF is trivially 1) and vary only the exponents. The variable part of the GCF is the **smallest exponent present** — the power of $x$ that every single term has "to spare."

A common instinct is to reach for the *largest* exponent instead, since that is the one that "stands out." That instinct is backwards — section 6 below shows exactly what goes wrong when you follow it.

In [ ]:
show_factor({5: 1, 3: 1, 2: 1})   # smallest exponent = 2
show_factor({4: 1, 3: 1, 1: 1})   # smallest exponent = 1


---

## 5. Isolating the Sign Convention

One more dimension, independent of the first two: what happens when the leading term is negative. The convention is to factor out a negative GCF so that the remaining polynomial's leading term is positive — this makes the result easier to read and to factor further in later notebooks.

In [ ]:
show_factor({3:  6, 2: 9})    # positive leading term -- GCF stays positive
show_factor({3: -6, 2: -9})   # negative leading term -- GCF flips negative, remaining stays positive-leading


---

## 6. Two Ways to Get This Wrong

### Non-example A — the reversed instinct (buggy code)

Section 4 warned about reaching for the *largest* exponent instead of the smallest. Here is exactly that bug, written the way it would actually be written — a one-word swap, `max` instead of `min`:

In [ ]:
def buggy_poly_gcf(poly):
    coeffs = list(poly.values())
    exps = list(poly.keys())
    g_coeff = coeff_gcf(coeffs)
    g_exp = max(exps)   # BUG: should be min(exps)
    return g_coeff, g_exp

poly = {3: 12, 2: 18, 1: 24}
g_coeff, g_exp = buggy_poly_gcf(poly)
remaining = poly_div_mono(poly, g_coeff, g_exp)
reconstructed = poly_mul_mono(remaining, g_coeff, g_exp)

print(f'Buggy attempt: {mono_str(g_coeff, g_exp)}({poly_str(remaining)})')
print(f'Remaining polynomial dict: {remaining}')
print(f'Reconstructed: {poly_str(reconstructed)}   (original: {poly_str(poly)})')
print(f'Numeric check passes: {poly_eval(poly, 5) == poly_eval(reconstructed, 5)}')


The numeric check *passes* — plug in $x = 5$ and both sides agree. But look at `remaining`: it has **negative exponents**. $3x^{-1}$ and $4x^{-2}$ are not polynomial terms at all. The bug produced something that re-multiplies correctly but is not a valid factorization, because it isn't a polynomial on one side of the equals sign anymore.

This is the trap with numeric verification: it catches arithmetic errors, but it does not catch "this isn't the kind of answer the question asked for." Checking that every resulting exponent is a non-negative integer is part of confirming a GCF factoring is actually valid — not just that the arithmetic checks out.

### Non-example B — technically true, but not the *greatest* common factor

In [ ]:
poly = {3: 12, 2: 18, 1: 24}
partial_coeff, partial_exp = 3, 1   # 3x divides every term -- but it isn't the GREATEST common factor
remaining = poly_div_mono(poly, partial_coeff, partial_exp)
reconstructed = poly_mul_mono(remaining, partial_coeff, partial_exp)

print(f'Partial factoring: {mono_str(partial_coeff, partial_exp)}({poly_str(remaining)})')
print(f'Verify at x=5: {poly_eval(poly, 5)} = {poly_eval(reconstructed, 5)}  '
      f'({"OK" if poly_eval(poly,5)==poly_eval(reconstructed,5) else "MISMATCH"})')

# The tell: does the remaining polynomial still share a common factor of its own?
remaining_ints = {e: int(c) for e, c in remaining.items()}
still_common = coeff_gcf(list(remaining_ints.values()))
print(f'Remaining polynomial {poly_str(remaining)} still has coefficient GCF = {still_common}')
print(f'-> {mono_str(partial_coeff, partial_exp)} was A common factor, but not THE greatest one.')


`3x` divides every term of $12x^3 + 18x^2 + 24x$ evenly, and the check at $x = 5$ passes — this is a valid factoring, just not the *complete* one. The tell: the leftover polynomial, $4x^2 + 6x + 8$, still shares a common factor of 2 among its own coefficients. **If the remaining polynomial's own coefficient-GCF is greater than 1, you stopped too early.** The true GCF, $6x$, is the one that leaves $2x^2 + 3x + 4$ behind — a polynomial with nothing left to pull out.

---

## 7. Generalizing — New Shapes, Same Idea

Two structural variations that haven't shown up yet, to confirm the pattern holds beyond the three-term, no-constant examples used so far.

**A constant term present.** If one of the terms has no $x$ at all (exponent 0), the smallest exponent across all terms is 0 — meaning **no power of $x$ can be factored out at all**, even if every other term has one.

In [ ]:
show_factor({2: 4, 0: 4})       # constant term present -> variable GCF is x^0 = 1 (no x factors out)
show_factor({3: 6, 1: 6, 0: 6})  # same idea with three terms


**More terms.** Nothing about the GCF process depends on there being exactly three terms — it works identically with four (or more) terms in the polynomial.

In [ ]:
show_factor({4: 8, 3: 12, 2: 4, 1: 20})   # four terms


---

## 8. Fusion — Combining Every Dimension at Once

Now that each dimension — numeric GCF, variable GCF, sign, constant terms, term count — has been seen in isolation, here is one example that combines several of them at once: a negative leading coefficient, a constant term (so no $x$ factors out), and four terms.

In [ ]:
show_factor({3: -6, 2: -9, 1: 3, 0: -12})


---

## 9. Visualisation — Factoring Reverses Multiplication

Section 1's tiling picture showed the numeric side. This shows the whole operation: the expanded polynomial on one side, the factored form on the other, with the same GCF appearing as a single shared block being pulled out of every term.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('12x^3 + 18x^2 + 24x   <-- factor / multiply -->   6x(2x^2 + 3x + 4)', fontsize=12, fontweight='bold')

term_colors = ['#89dceb', '#a6e3a1', '#fab387']
terms = [(3, 12), (2, 18), (1, 24)]

# ---- Left: expanded form -- three separate colored blocks ----
ax = axes[0]
ax.set_title('Expanded: three independent terms', fontsize=10)
ax.set_xlim(0, 3)
ax.set_ylim(0, 26)
ax.axis('off')
for i, (exp, coeff) in enumerate(terms):
    rect = mpatches.FancyBboxPatch((i + 0.1, 0), 0.8, coeff, boxstyle='round,pad=0.02',
                                    facecolor=term_colors[i], edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(i + 0.5, coeff + 1, mono_str(coeff, exp), ha='center', fontsize=11, fontweight='bold')

# ---- Right: factored form -- shared 6x block, times a bracket of what's left ----
ax2 = axes[1]
ax2.set_title('Factored: one shared block, pulled out of all three', fontsize=10)
ax2.set_xlim(0, 3)
ax2.set_ylim(0, 26)
ax2.axis('off')
remaining_coeffs = [2, 3, 4]   # 2x^2 + 3x + 4, after dividing out 6x
for i, (rc, orig_color) in enumerate(zip(remaining_coeffs, term_colors)):
    shared = mpatches.FancyBboxPatch((i + 0.1, 0), 0.8, 6, boxstyle='round,pad=0.02',
                                      facecolor='#cba6f7', edgecolor='white', linewidth=2)
    rest = mpatches.FancyBboxPatch((i + 0.1, 6.3), 0.8, rc * 6, boxstyle='round,pad=0.02',
                                    facecolor=orig_color, edgecolor='white', linewidth=2, alpha=0.5)
    ax2.add_patch(shared)
    ax2.add_patch(rest)
    ax2.text(i + 0.5, 3, '6x', ha='center', va='center', fontsize=10, fontweight='bold')
ax2.text(1.5, 24, '(2x^2 + 3x + 4)', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


---

## Exercises

Work each on paper first — find the GCF, factor it out, then verify in code.

1. Factor $8x^3 + 12x^2$.
2. Factor $15x^4 - 25x^3 + 10x^2$.
3. Factor $x^5 + x^4 + x^3$. What is the variable part of the GCF here?
4. Factor $-4x^3 - 6x^2 + 2x$. Watch the sign convention.
5. Factor $9x^2 + 9$. Why doesn't any power of $x$ factor out?
6. Factor $6x^4 + 9x^3 - 3x^2 + 12x$ (four terms).
7. A classmate factors $10x^3 + 15x^2$ as $5x(2x^2 + 3x)$. Check their work: is $5x$ actually the GCF? If not, what is, and what's left over as a tell?
8. A classmate writes code using `max(exps)` for the variable part and gets `2x^-1 + 3` as their "remaining polynomial." What does that output tell you about their code, before you even check the arithmetic?
9. Factor $-x^3 - x^2 - x$. What is the GCF, including its sign?
10. Factor $7x^5 - 7x^3$. Verify by multiplying your factored form back out.

In [ ]:
# Verify your answers here.
# poly_gcf(poly)            -- returns (gcf_coeff, gcf_exp)
# factor_gcf(poly)          -- returns (gcf_coeff, gcf_exp, remaining_poly)
# show_factor(poly)         -- prints the whole process with verification
# poly_str(p), poly_eval(p, x), mono_str(coeff, exp)

# Example: exercise 1
# show_factor({3: 8, 2: 12})


---

## Formal Notation

Let $P(x) = \sum_{i} a_i x^i$ be a polynomial with nonzero coefficients $a_i$ at exponents $i \in S$ (the set of degrees actually present).

The **greatest common factor** of $P$'s terms is the monomial:

$$\mathrm{GCF}(P) = c \cdot x^{k}, \qquad c = \gcd(|a_i| : i \in S), \qquad k = \min(S)$$

with $c$ given the same sign as $-1$ times the sign of the leading coefficient when that leading coefficient is negative (the convention that keeps the remaining polynomial's leading term positive).

Factoring out the GCF gives:

$$P(x) = \mathrm{GCF}(P) \cdot Q(x)$$

where $Q(x) = P(x) / \mathrm{GCF}(P)$, applied term-by-term (notebook 18, section 5). $Q$ is **fully reduced** with respect to this operation when $\gcd$ of $Q$'s own coefficients is 1 and $Q$ has a nonzero constant term or $\min(S_Q) = 0$ — i.e., nothing further can be pulled out of every term of $Q$ the same way.

This is structurally identical to reducing a fraction $\frac{a}{b}$ to lowest terms by dividing both $a$ and $b$ by $\gcd(a, b)$ (notebook 04) — GCF factoring is that same reduction, applied to a polynomial's terms instead of a fraction's numerator and denominator.

---

## Real-World Connection

- **Simplifying before further factoring:** GCF extraction is always the first step checked in any factoring algorithm — computer algebra systems (SymPy, Mathematica) run a GCF pass before attempting trinomial factoring, grouping, or any other technique, for the same reason you would: it makes everything downstream simpler and sometimes reveals the answer outright.

- **Reducing fractions:** `fractions.Fraction` in Python automatically reduces `6/9` to `2/3` using `math.gcd` internally — literally the `coeff_gcf` helper used in this notebook, applied to a numerator and denominator instead of a list of polynomial coefficients.

- **Choosing a common block size:** image and video codecs (JPEG's 8×8 blocks, video codecs' macroblocks) need a block size that evenly divides the frame's width and height. Picking that size is a GCF problem — the largest block that tiles a width and a height with nothing left over.

- **Signal processing:** finding a shared sampling interval that evenly divides several signal periods, so that multiple signals can be sampled on the same timeline without misalignment, is the GCF of the periods.

- **Aligning memory and vector widths:** compilers choose loop-unrolling and vectorization strides based on the GCF of a data structure's size and the target hardware's vector width, so that loops divide evenly with no ragged leftover iteration.

---

## Summary

**Factoring out the GCF is multiplication in reverse.** Given an expanded polynomial, find the largest monomial that could have been distributed to produce it, and pull it back out.

- **Numeric part:** the greatest common factor of the coefficients — `gcd` applied across every term, exactly as when reducing a fraction to lowest terms.
- **Variable part:** the *smallest* exponent present across every term — the power of $x$ every term can spare. Reaching for the largest exponent instead is a common, specific bug: it produces negative exponents in the "remaining" polynomial, which is a sign that something isn't a valid polynomial factorization anymore, even if the arithmetic still checks out numerically.
- **Sign convention:** factor out a negative GCF when the leading term is negative, so the remaining polynomial's leading term is positive.
- **"Greatest" matters:** a factor that divides every term evenly but isn't the *largest* such factor is a valid but incomplete factoring. The tell: the remaining polynomial still shares a common factor among its own coefficients.
- **Verification:** multiply the GCF back into the remaining polynomial and confirm it reproduces the original — and additionally confirm every resulting exponent is a non-negative integer, since numeric equality alone doesn't guarantee a valid factorization.